# 02 — Visualise InSAR Products
**Urban Subsidence Watcher** | Samuel Appiah Kubi | PLUS Salzburg

This notebook visualises the InSAR interferograms, coherence maps, unwrapped
phase, and displacement time series produced by the SBAS pipeline.

**Pre-requisites:** Run `python main.py --step insar` before opening this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import rasterio
import json
import yaml
import warnings
warnings.filterwarnings('ignore')

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

DATA = Path('../data')
INSAR_DIR = DATA / 'insar'

print(f"InSAR directory: {INSAR_DIR}")
print(f"Exists: {INSAR_DIR.exists()}")

## 1. SBAS Interferogram Network

In [ ]:
network_path = INSAR_DIR / 'network.json'

if network_path.exists():
    with open(network_path) as f:
        network = json.load(f)
    pairs = network['pairs']
    tb = network['temporal_baselines_days']
    print(f'Network: {len(pairs)} interferograms from {network["n_epochs"]} epochs')
else:
    from datetime import datetime, timedelta
    print('[INFO] network.json not found — generating synthetic SBAS network for demo.')
    base_date = datetime(2020, 1, 1)
    n_epochs = 20
    dates = [base_date + timedelta(days=12*i) for i in range(n_epochs)]
    pairs = []
    tb = []
    for i in range(n_epochs):
        for j in range(i+1, n_epochs):
            delta = (dates[j] - dates[i]).days
            if delta <= 120:
                pairs.append([dates[i].strftime('%Y-%m-%d'), dates[j].strftime('%Y-%m-%d')])
                tb.append(delta)
    network = {'pairs': pairs, 'temporal_baselines_days': tb, 'n_epochs': n_epochs, 'n_ifgrams': len(pairs)}

# Plot SBAS network
from datetime import datetime

dates_a = [datetime.strptime(p[0], '%Y-%m-%d') for p in pairs]
dates_b = [datetime.strptime(p[1], '%Y-%m-%d') for p in pairs]

fig, ax = plt.subplots(figsize=(14, 5))
for da, db, t in zip(dates_a, dates_b, tb):
    ax.plot([da, db], [0, 0], '-', alpha=0.4, color=cm.viridis(t/120), lw=1.5)
    ax.plot([da, db], [0, 0], 'o', ms=3, color='steelblue')

# Epoch markers
all_dates = sorted(set(dates_a + dates_b))
ax.plot(all_dates, [0]*len(all_dates), 'k^', ms=6, zorder=5, label='Acquisition dates')

sm = plt.cm.ScalarMappable(cmap=cm.viridis, norm=plt.Normalize(0, 120))
plt.colorbar(sm, ax=ax, label='Temporal baseline (days)')
ax.set_yticks([])
ax.set_xlabel('Date', fontsize=12)
ax.set_title(f'SBAS Interferogram Network — {len(pairs)} interferograms, {len(all_dates)} epochs', fontsize=13)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(DATA / 'validation' / 'sbas_network.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Interferogram Gallery

In [ ]:
ifg_dir = INSAR_DIR / 'interferograms'
ifg_files = sorted(ifg_dir.glob('*.tif')) if ifg_dir.exists() else []

n_show = min(6, len(ifg_files))

if n_show > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, f in zip(axes.ravel(), ifg_files[:n_show]):
        with rasterio.open(f) as src:
            data = src.read(1)
        im = ax.imshow(np.angle(data.astype(complex)), cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title(f.stem[:20], fontsize=8)
        ax.axis('off')
    plt.colorbar(im, ax=axes.ravel().tolist(), label='Phase (rad)')
    plt.suptitle('InSAR Interferograms — Wrapped Phase', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] No interferogram files found — generating 6 synthetic interferograms for demo.')
    rng = np.random.default_rng(7)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, ax in enumerate(axes.ravel()):
        phase = rng.uniform(-np.pi, np.pi, (80, 80)).astype(np.float32)
        # Add a subsidence signal (concentric fringes)
        y, x = np.mgrid[-40:40, -40:40]
        signal = 0.5 * np.sin(np.sqrt(x**2 + y**2) / 5)
        phase = np.angle(np.exp(1j * (phase * 0.3 + signal)))
        im = ax.imshow(phase, cmap='hsv', vmin=-np.pi, vmax=np.pi)
        ax.set_title(f'IFG {i+1:02d} (synthetic)', fontsize=8)
        ax.axis('off')
    plt.colorbar(im, ax=axes.ravel().tolist(), label='Phase (rad)')
    plt.suptitle('InSAR Interferograms — Wrapped Phase (Synthetic Demo)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 3. Coherence Maps

In [ ]:
coh_dir = INSAR_DIR / 'coherence'
coh_files = sorted(coh_dir.glob('*.tif')) if coh_dir.exists() else []

n_show = min(6, len(coh_files))

if n_show > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, f in zip(axes.ravel(), coh_files[:n_show]):
        with rasterio.open(f) as src:
            coh = src.read(1)
        im = ax.imshow(coh, cmap='magma', vmin=0, vmax=1)
        ax.set_title(f.stem[:20], fontsize=8)
        ax.axis('off')
    plt.colorbar(im, ax=axes.ravel().tolist(), label='Coherence')
    plt.suptitle('Coherence Maps', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] No coherence files found — synthetic demo.')
    rng = np.random.default_rng(13)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, ax in enumerate(axes.ravel()):
        coh = np.clip(rng.beta(3, 2, (80, 80)).astype(np.float32), 0, 1)
        im = ax.imshow(coh, cmap='magma', vmin=0, vmax=1)
        ax.set_title(f'Coherence {i+1:02d} (synthetic)', fontsize=8)
        ax.axis('off')
    plt.colorbar(im, ax=axes.ravel().tolist(), label='Coherence')
    plt.suptitle('Coherence Maps (Synthetic Demo)', fontsize=13)
    plt.tight_layout()
    plt.show()

## 4. Displacement Time Series (SBAS)

In [ ]:
ts_dir = INSAR_DIR / 'timeseries'
ts_files = sorted(ts_dir.glob('displacement_*.tif')) if ts_dir.exists() else []

if len(ts_files) > 0:
    # Extract time series for the centre pixel
    displacements = []
    for f in ts_files:
        with rasterio.open(f) as src:
            arr = src.read(1)
        displacements.append(arr[arr.shape[0]//2, arr.shape[1]//2])
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(range(len(displacements)), displacements, 'o-', color='steelblue', ms=4)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Displacement (mm)')
    ax.set_title('SBAS Displacement Time Series — Centre Pixel')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] Time-series rasters not found — synthetic demo.')
    rng = np.random.default_rng(99)
    n_epochs = 30
    t = np.arange(n_epochs)
    
    fig, ax = plt.subplots(figsize=(12, 4))
    for rate_val, label in [(-15, 'Hotspot A'), (-5, 'Hotspot B'), (-25, 'Hotspot C')]:
        noise = rng.normal(0, 0.8, n_epochs).cumsum() * 0.3
        d = rate_val / 365 * t * 12 + noise  # 12-day repeat
        ax.plot(t, d, 'o-', ms=4, label=f'{label} ({rate_val} mm/yr)')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Displacement (mm)')
    ax.set_title('SBAS Displacement Time Series — Three Pixels (Synthetic Demo)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Model Predictions vs InSAR

In [ ]:
# Compare AI model prediction with InSAR-derived rate
rate_path = DATA / 'subsidence' / 'rate_map.tif'
pred_path = DATA / 'forecasts' / 'forecast_1yr.tif'

if rate_path.exists() and pred_path.exists():
    with rasterio.open(rate_path) as src:
        insar_rate = src.read(1)
    with rasterio.open(pred_path) as src:
        model_pred = src.read(1)
    title = 'InSAR vs Model'
else:
    rng = np.random.default_rng(55)
    insar_rate = rng.uniform(-30, 2, (100, 100)).astype(np.float32)
    model_pred = insar_rate + rng.normal(0, 2, (100, 100)).astype(np.float32)
    title = 'InSAR vs Model (Synthetic Demo)'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(title, fontsize=13, fontweight='bold')

kw = dict(cmap='RdYlGn', vmin=-30, vmax=5)
axes[0].imshow(insar_rate, **kw)
axes[0].set_title('InSAR Rate (mm/yr)')

axes[1].imshow(model_pred, **kw)
axes[1].set_title('Model Prediction (mm/yr)')

diff = model_pred - insar_rate
im = axes[2].imshow(diff, cmap='bwr', vmin=-5, vmax=5)
axes[2].set_title('Residual (Model − InSAR)')
plt.colorbar(im, ax=axes[2], label='mm/yr')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

# Scatter
valid_mask = np.isfinite(insar_rate) & np.isfinite(model_pred)
x = insar_rate[valid_mask].ravel()
y = model_pred[valid_mask].ravel()

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(x, y, alpha=0.1, s=1, c='steelblue')
lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
ax.plot([lo, hi], [lo, hi], 'r--', label='1:1 line')
mae = np.mean(np.abs(x - y))
ax.set_xlabel('InSAR Rate (mm/yr)')
ax.set_ylabel('Model Prediction (mm/yr)')
ax.set_title(f'Scatter: MAE = {mae:.2f} mm/yr')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()